In [1]:
import sys
import os

__file__ = "/Users/jmfrutos/github/ISL-python/ISL/examples"
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from ISL.isl import ts_invariant_statistical_loss, ts_invariant_statistical_loss_2, CondTimeGenModel
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Normal


In [3]:
class RecurrentRNN(nn.Module):
    def __init__(self, input_size, hidden_size, window_size):
        super(RecurrentRNN, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, window_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out

class GeneratorNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(GeneratorNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Example hyperparameters
hparams = {
    'eta': 1e-2,
    'window_size': 100,
    'K': 5
}

# Initialize models
rec = RecurrentRNN(input_size=1, hidden_size=64, window_size=hparams['window_size'])
gen = GeneratorNN(input_size=hparams['window_size'] + 1, output_size=1)

model = CondTimeGenModel(rec, gen)

In [4]:
import pandas as pd

# Define the path to your file
file_path = '/Users/jmfrutos/github/ISL-python/data/LD2011_2014.txt'

# Load the data
# Assuming the first row contains headers and the first column contains datetime information
data = pd.read_csv(file_path, sep=';', index_col=0, parse_dates=True, decimal=',')

In [5]:
first_non_zero_index_MT_001 = data['MT_001'].ne(0).idxmax()
X_t = data['MT_001'][first_non_zero_index_MT_001:] # Your time series data
Y_t = data['MT_001'][data['MT_001'].index > first_non_zero_index_MT_001] # Your shifted time series data by one time step


In [6]:
noise_model = Normal(0.0, 1.0)
hparams = {
    'max_k': 10,
    'samples': 10000,
    'epochs': 1000,
    'eta': 1e-2,
    'transform': noise_model,
    'window_size': 100,
    'K': 5
}

# Train the model using the ts_invariant_statistical_loss function
loader_Xt = DataLoader(X_t, batch_size=hparams['samples'], shuffle=False)
loader_Yt = DataLoader(Y_t, batch_size=hparams['samples'], shuffle=False)

ts_invariant_statistical_loss_2(model, loader_Xt, loader_Yt, hparams)

0it [00:00, ?it/s]

0
Gradient for seq_model.rnn.weight_ih_l0: tensor([[-1.0269e-22],
        [-2.8287e-23],
        [-4.4359e-24],
        [ 3.4299e-23],
        [-7.9335e-23],
        [ 8.4734e-23],
        [-4.3887e-23],
        [ 6.0854e-24],
        [ 5.5333e-23],
        [ 4.2307e-24],
        [-1.0073e-23],
        [ 1.7377e-23],
        [ 4.0496e-24],
        [-2.6352e-22],
        [ 2.6783e-22],
        [-2.5813e-24],
        [ 1.5755e-22],
        [-2.2418e-23],
        [ 3.3074e-23],
        [-2.2432e-22],
        [-1.1029e-22],
        [-7.3773e-23],
        [ 1.2307e-22],
        [-2.6787e-23],
        [-3.2855e-23],
        [ 3.4509e-22],
        [ 1.1101e-23],
        [ 2.6659e-22],
        [ 1.9367e-23],
        [ 2.1349e-22],
        [ 2.7540e-24],
        [-3.1053e-22],
        [ 3.3984e-23],
        [-8.8493e-24],
        [-8.0634e-24],
        [ 2.8641e-23],
        [ 6.1983e-24],
        [ 8.2430e-24],
        [ 5.0886e-23],
        [-1.3560e-23],
        [ 1.4882e-23],
        [-3.85

In [ ]:
iterator_Xt = next(iter(loader_Xt))

xt = iterator_Xt.float()

#xt.view(1,1,-1)

#rec(xt.view(1,1,-1))

#xt.view(1,1,-1)


#rec()

#data = t.view(10, 10, -1) 

#rec(xt)
len(iterator_Xt[0:0 + hparams['window_size']])

h0 = torch.zeros(1, 1, 100)  # Shape: (num_layers * num_directions, batch, hidden_size)
c0 = torch.zeros(1, 100, 1)

#model.seq_model(iterator_Xt[0:0 + hparams['window_size']], (h_0, c_0))

#model.seq_model
#rec(iterator_Xt[0:0 + hparams['window_size']].transpose(0,0))
#model.generated_fictitious(iterator_Xt[0:0 + hparams['window_size']], hparams['K'])
#model.seq_model()
#model.seq_model(iterator_Xt[0:0 + hparams['window_size']], c_0)

#rec(iterator_Xt[0:0 + hparams['window_size']].unsqueeze(1).unsqueeze(2).float())

print(iterator_Xt[0:0 + hparams['window_size']].unsqueeze(1).unsqueeze(-1).float())

rnn = nn.RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)
output, hn = rnn(input, h0)